# RAG Knowledge Base - Corpus Analysis (EDA)
## The retrieval corpus behind the Hybrid Medical Assistant

This notebook is the exploratory data analysis for the **RAG pillar's** knowledge base:
the chunked medical literature in `data/processed/passages.jsonl` that the retriever
searches and that every grounded LLM answer is cited from.

### Why this dataset spans both RAG and LLM
The project has three pillars - a free-text ML classifier, RAG retrieval, and an LLM.
The LLM (Ollama) is **not** fine-tuned on any local dataset; it is instructed to answer
*only* from retrieved passages. So this corpus is simultaneously the RAG search index and
the sole grounding source for the LLM. Its coverage and skew directly shape what the
assistant can and cannot answer well.

### Companion notebooks (already in the repo)
- `notebooks/text_classifier_analysis.ipynb` - EDA + training for the live free-text
  symptom classifier (`gretelai/symptom_to_diagnosis`).
- `notebooks/ml_model_analysis.ipynb` - EDA + training for the legacy DDXPlus XGBoost.
- `models/training/{mri,eeg,ecg}.ipynb` - the imaging/signal models (each contains its own
  class-distribution EDA and sample plots).

This notebook fills the remaining gap: the RAG/LLM knowledge corpus.

## 1. Environment setup

In [ ]:
import json
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4)
pd.set_option("display.max_colwidth", 90)

## 2. Loading the corpus

Each line of `passages.jsonl` is one retrievable chunk produced by `rag/ingest.py`
(parse -> clean -> chunk). We load it into a DataFrame for analysis.

In [ ]:
# Resolve the path whether the notebook runs from repo root or notebooks/
candidates = [Path("data/processed/passages.jsonl"),
              Path("../data/processed/passages.jsonl")]
PASSAGES = next(p for p in candidates if p.exists())

rows = [json.loads(l) for l in PASSAGES.open(encoding="utf-8") if l.strip()]
df = pd.DataFrame(rows)
print(f"Loaded {len(df):,} passages from {PASSAGES}")
df.head(3)

### Data schema

| Field | Meaning |
|-------|---------|
| `id` | Stable chunk id (`<doc>__<chunk index>`) |
| `title` | Source article title (usually a condition name) |
| `question` | The MedQuAD Q&A question the passage answers |
| `qtype` | Question category (symptoms, treatment, causes, ...) |
| `text` | The passage body - what gets embedded and retrieved |
| `source` | Originating sub-corpus (GARD, GHR, CancerGov, ...) |
| `url` | Canonical source URL (shown as a citation) |

In [ ]:
print("Columns:", list(df.columns))
print("Null counts:")
print(df.isna().sum())

## 3. Source distribution - the rare-disease skew

The single most important property of this corpus: it is **dominated by rare-disease
references**. GARD (Genetic and Rare Diseases) and GHR (Genetics Home Reference) together
make up ~60% of all passages, while everyday-health sources (MedlinePlus, CDC) are thin.

In [ ]:
src = df["source"].value_counts()
pct = (src / len(df) * 100).round(1)
summary = pd.DataFrame({"passages": src, "pct": pct})
display(summary)

ax = src.plot(kind="bar", color=sns.color_palette("viridis", len(src)))
ax.set_title("Passages per source"); ax.set_ylabel("passages"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

rare = df["source"].isin(["GARD", "GHR"]).mean() * 100
print(f"Rare-disease sources (GARD + GHR): {rare:.1f}% of the corpus")

### Observations - source skew

- **GARD 30.8% + GHR 29.4% = ~60%** of the corpus is rare / genetic disease material.
- Everyday primary-care topics (MedlinePlus `MPlusHealthTopics` ~5%, `CDC` ~2%) are
  under-represented.
- **Retrieval implication:** the assistant is strongest on named rare/genetic conditions
  and cancers, and weakest on common complaints (colds, minor injuries, routine wellness).
  The confidence gate (`RERANK_SCORE_FLOOR`) is what stops it from fabricating an answer
  when a common-topic query finds no well-matched passage - it declines instead.

## 4. Question-type (qtype) coverage

MedQuAD passages are organised by the type of question they answer. This shows *what kind*
of information the corpus is rich in.

In [ ]:
qt = df["qtype"].value_counts()
display(qt.to_frame("passages"))

ax = qt.head(12).plot(kind="barh", color=sns.color_palette("crest", 12))
ax.invert_yaxis(); ax.set_title("Question types (top 12)"); ax.set_xlabel("passages")
plt.tight_layout(); plt.show()

### Observations - qtype

- `information`, `symptoms`, and `treatment` dominate - a good fit for a symptom-exploration
  assistant that explains conditions and next steps.
- Genetics-flavoured types (`inheritance`, `genetic changes`, `susceptibility`) are
  prominent, echoing the GARD/GHR skew above.

## 5. Document & chunking structure

Passages are chunks of larger source documents. Here we reconstruct the document level
(the part of the id before the final `__chunk`) to see how aggressively documents were split.

In [ ]:
def doc_key(r):
    i = r.get("id", "")
    return i.rsplit("__", 1)[0] if "__" in i else (r.get("url") or r.get("title"))

df["doc"] = df.apply(doc_key, axis=1)
per_doc = df.groupby("doc").size()

print(f"Unique documents : {per_doc.size:,}")
print(f"Passages / doc   : mean {per_doc.mean():.2f}, median {per_doc.median():.0f}, max {per_doc.max()}")

ax = per_doc.value_counts().sort_index().plot(kind="bar", color="steelblue")
ax.set_title("Passages per document"); ax.set_xlabel("chunks in a document"); ax.set_ylabel("documents")
plt.tight_layout(); plt.show()

### Observations - chunking

- Most documents are short and become a **single chunk** (median 1, mean ~1.2), so a
  retrieved passage usually is the whole article - good for self-contained citations.
- A long tail of documents split into up to ~20 chunks (large cancer / disease overviews).

## 6. Passage length distribution

Chunk length drives both embedding quality and how much context each retrieved passage
contributes to the LLM prompt.

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
display(df["n_words"].describe().round(1).to_frame("words"))

ax = df["n_words"].plot(kind="hist", bins=40, color="mediumpurple", edgecolor="white")
ax.set_title("Passage length (words)"); ax.set_xlabel("words per passage")
plt.tight_layout(); plt.show()

print("Near-empty passages (<5 words):", int((df["n_words"] < 5).sum()))

### Observations - length

- Lengths cluster around a **median of ~149 words** with a cap near 450 (the chunker's
  target), so passages are uniform enough for stable embeddings.
- Only a handful of near-empty passages exist - negligible noise.

## 7. Topic coverage - most-represented conditions

Which conditions have the most material? This is a proxy for where the assistant is
deepest.

In [ ]:
top_titles = df["title"].value_counts().head(15)
display(top_titles.to_frame("passages"))
print("Unique titles (conditions/topics):", df["title"].nunique())

ax = top_titles.plot(kind="barh", color=sns.color_palette("flare", 15))
ax.invert_yaxis(); ax.set_title("Top 15 topics by passage count"); ax.set_xlabel("passages")
plt.tight_layout(); plt.show()

### Observations - coverage

- The densest everyday topics are the big cancers and cardiovascular/metabolic conditions
  (Breast/Prostate/Skin Cancer, Stroke, High Blood Pressure, Diabetes) - these are the
  common-health areas the corpus *does* cover well.
- With **~5,000 unique titles**, breadth is high, but depth per topic is shallow outside
  the head (mean ~1.2 passages per document).

## 8. Retrieval & grounding implications

Tying the EDA back to the running system:

1. **Skew -> declining gracefully.** Because common complaints are thin, the retrieval
   confidence gate and topicality check are load-bearing: on an out-of-corpus query the
   assistant returns a 'no relevant info' refusal rather than an ungrounded answer.
2. **Rare/genetic strength.** For named rare or genetic conditions the corpus is unusually
   rich, so retrieval recall is high and citations are specific.
3. **Uniform chunks -> stable citations.** Single-chunk documents mean a citation usually
   points at a whole coherent article.
4. **Expansion lever.** To improve everyday-health usefulness, the highest-value ingest
   would be more MedlinePlus / CDC primary-care material to rebalance the ~60% rare-disease
   share (a heavy re-ingest via `python -m rag.ingest`).

## 9. Summary & limitations

**What this corpus is:** ~18,900 passages / ~15,500 documents of authoritative U.S.
government / NIH medical literature (GARD, GHR, CancerGov, NIDDK, NINDS, NHLBI, MedlinePlus,
CDC, NIHSeniorHealth), chunked for retrieval.

**Strengths:** trustworthy sources, broad condition coverage (~5k topics), uniform
self-contained chunks, strong on rare/genetic disease and major cancers.

**Limitations (honest - this matters for a medical tool):**
- ~60% rare-disease skew; everyday primary-care coverage is thin.
- Depth is shallow outside the head topics (~1.2 passages/doc).
- English-only, U.S.-centric, and static (a snapshot - no live guideline updates).
- No pediatric-specific or medication-interaction depth.

These are exactly why the system is framed as **educational, non-diagnostic**, and why the
grounding gate exists: the corpus is a strong but bounded knowledge base, and the assistant
is designed to decline rather than guess when a question falls outside it.